# Lab 02.4 – Button-Controlled PWM

## Objectives
- Combine push-button input with PWM output.
- Increase and decrease LED brightness interactively.
- Apply edge detection to avoid repeated actions while a button is held.
- Use a simple software debounce delay.

## Controls
- **BTN0**: increase duty cycle by 10%
- **BTN1**: decrease duty cycle by 10%
- **BTN2**: set duty cycle to 50%
- **BTN3**: stop the program


In [ ]:
from pynq import Overlay
import time

ol = Overlay("base.bit")

buttons = ol.btns_gpio.channel1
buttons.setdirection("in")
buttons.setlength(4)

leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

print("GPIO initialized.")


## Exercise – Interactive brightness control

The program continuously generates short PWM bursts and checks the buttons between bursts.


In [ ]:
def pwm_burst(duty_cycle, frequency=100, cycles=5):
    duty_cycle = max(0.0, min(100.0, duty_cycle))

    period = 1.0 / frequency
    on_time = period * duty_cycle / 100.0
    off_time = period - on_time

    for _ in range(cycles):
        if on_time > 0:
            leds.write(0b0001, 0b1111)
            time.sleep(on_time)

        if off_time > 0:
            leds.write(0b0000, 0b1111)
            time.sleep(off_time)


In [ ]:
duty = 50
previous = 0

print(f"Duty cycle = {duty}%")

while True:
    pwm_burst(duty, frequency=100, cycles=5)

    current = buttons.read() & 0xF
    pressed = current & ~previous

    if pressed & 0b0001:          # BTN0
        duty = min(100, duty + 10)
        print(f"Duty cycle = {duty}%")

    if pressed & 0b0010:          # BTN1
        duty = max(0, duty - 10)
        print(f"Duty cycle = {duty}%")

    if pressed & 0b0100:          # BTN2
        duty = 50
        print(f"Duty cycle = {duty}%")

    if pressed & 0b1000:          # BTN3
        print("Program stopped.")
        break

    if pressed:
        time.sleep(0.05)

    previous = current

leds.write(0, 0b1111)


## Questions
1. Why is `pressed = current & ~previous` used?
2. Why are the duty-cycle limits restricted to 0...100%?
3. What happens at 0% and 100% duty cycle?
4. Why is this implementation suitable for demonstration but not for precise motor control?
5. How could the PWM generation be moved from software to the FPGA fabric?
